In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from scipy.ndimage import gaussian_filter
from utils import *

In [2]:
folder = 'temp/'
files = sorted(glob.glob(folder + '*.fits'))
files

['temp/phi-fdt-flat_20260310T040003_V202607131606C_0663100100.fits',
 'temp/phi-fdt-ghost_20260310T040003_V202607131606C_0663100100.fits']

In [3]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
flat_file, ghost_file = files

with fits.open(dark_file) as hdul:
    dark = hdul[0].data

with fits.open(flat_file) as hdul:
    flat = hdul[0].data

with fits.open(ghost_file) as hdul:
    ghost = hdul[0].data

ghost = demodulate(ghost)

In [4]:
folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T040003_V202606241235C_0663100100.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T040405_V202606241235C_0663100125.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T040805_V202606241336C_0663100150.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T041205_V202606241336C_0663100175.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T041605_V202606241434C_0663100200.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T042005_V202606241434C_0663100225.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T042405_V202606241535C_0663100250.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026

In [5]:
with fits.open(files[0]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

cpos = header['CONTPOS'] - 1
data = data.reshape(6,4,2048,2048)[cpos]

data -= dark * 0.4
data /= flat
data = realign(data)
data = demodulate(data)

xr, yr = reflection_point_predict(header)
reflection = reflect(gaussian_filter(data[0], 8), xr, yr)

In [10]:
plt.figure(figsize=(10,10))
plt.imshow(data[0] - ghost[0] * reflection, vmin=-200, vmax=200)
plt.tight_layout()

In [11]:
i = 3

plt.figure(figsize=(10,10))
plt.imshow(data[i] - ghost[i] * reflection, vmin=-30, vmax=30)
plt.tight_layout()